In [9]:
# --------- 1 ------------

import numpy as np
import pandas as pd

from xgboost import XGBRegressor

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from sklearn.model_selection import KFold

In [10]:
# ---------- 2 ------------

df = pd.read_parquet("training_dataset_Bay_of_Bengal_clean.parquet")

# -----------------------------
# BOB
# Lon: 45°E–65°E
# Lat: 5°N–28°N
# -----------------------------

df = df[
    (df["longitude"] >= 78) &
    (df["longitude"] <= 100) &
    (df["latitude"]  >= 5) &
    (df["latitude"]  <= 25)
].copy()

print(f"Total samples in BOB = {len(df):,}")

TARGET = "chlor_a_log10"

DROP_COLS = [
    TARGET,
    "latitude",
    "longitude",
    "year",
    "month",
    "zos",
    "grad_ssh"
    
]

# -----------------------------
# Train : 1998-2024
# Test  : 2025
# -----------------------------

train_df = df[
    (df["year"] >= 1998) &
    (df["year"] <= 2024)
].copy()

test_df = df[
    df["year"] == 2025
].copy()

train_df = train_df.sort_values(
    ["year", "month"]
).reset_index(drop=True)

test_df = test_df.sort_values(
    ["year", "month"]
).reset_index(drop=True)

FEATURES = [
    col for col in df.columns
    if col not in DROP_COLS
]

X_train = train_df[FEATURES].astype("float32")
y_train = train_df[TARGET].astype("float32")

X_test = test_df[FEATURES].astype("float32")
y_test = test_df[TARGET].astype("float32")


X_train.head(3)

Total samples in BOB = 11,794,622


,month_sin,month_cos,so,thetao,uo,vo,mlotst_ln,vorticity,okubo_weiss
0,0.5,0.866025,32.884777,28.481493,-0.622853,0.013113,2.677283,1.760012e-07,2.478666e-12
1,0.5,0.866025,32.882660,28.480099,-0.624251,0.007324,2.674911,1.736737e-07,2.464752e-12
2,0.5,0.866025,32.881428,28.479034,-0.624350,0.001083,2.678973,1.904904e-07,2.574131e-12


In [11]:
# --------- 3 --------------

# ============================================================
# Standard K-Fold Cross Validation
# ============================================================

kf = KFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

fold_indices = list(kf.split(X_train))

# ============================================================
# Check Folds
# ============================================================

for i, (train_idx, valid_idx) in enumerate(fold_indices, start=1):

    print(f"\nFold {i}")

    print(
        f"Train Samples: {len(train_idx):,}"
    )

    print(
        f"Valid Samples: {len(valid_idx):,}"
    )


Fold 1
Train Samples: 5,626,057
Valid Samples: 5,626,057

Fold 2
Train Samples: 5,626,057
Valid Samples: 5,626,057


In [12]:
UNDER_PENALTY = 0.007

def underestimation_loss(y_true, y_pred):

    error = y_true - y_pred

    under_mask = error > 0

    grad = np.where(
        under_mask,
        -2.0 * (1.0 + UNDER_PENALTY) * error,
        -2.0 * error
    )

    hess = np.where(
        under_mask,
        2.0 * (1.0 + UNDER_PENALTY),
        2.0
    )

    return grad, hess

In [13]:
# ------------ 7 ---------------

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)



best_model_params = {

    

    "n_estimators":506,
    "max_depth":51,
    "learning_rate":0.022254925194568217,
    "subsample":0.9884760593039138,
    "colsample_bytree":0.963930837858085,
    "min_child_weight":44,
    "gamma":1.9006219720113502,
    "reg_alpha":0.002788121976650885,
    "reg_lambda":65.17811764107105 
}



results = []

for fold, (train_idx, valid_idx) in enumerate(
    fold_indices,
    start=1
):

    X_tr = X_train.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]

    X_val = X_train.iloc[valid_idx]
    y_val = y_train.iloc[valid_idx]

    model = XGBRegressor(
        objective=underestimation_loss,
        tree_method="gpu_hist",
        predictor="gpu_predictor",
        random_state=42,
        n_jobs=-1,
        **best_model_params
    )

    model.fit(
    X_tr,
    y_tr,
    verbose=False
    )
    
    
    

    

    

    # ----------------------------------                 # Train- val metrics 
    # Train Metrics
    # ----------------------------------

    pred_train = model.predict(X_tr)

    train_r2 = r2_score(y_tr, pred_train)

    train_rmse = np.sqrt(
        mean_squared_error(y_tr, pred_train)
    )

    train_mae = mean_absolute_error(
        y_tr,
        pred_train
    )

    train_bias = np.mean(
        pred_train - y_tr
    )

    # ----------------------------------
    # Validation Metrics
    # ----------------------------------

    pred_val = model.predict(X_val)

    val_r2 = r2_score(y_val, pred_val)

    val_rmse = np.sqrt(
        mean_squared_error(y_val, pred_val)
    )

    val_mae = mean_absolute_error(
        y_val,
        pred_val
    )

    val_bias = np.mean(
        pred_val - y_val
    )

    print(f"\nFold {fold}")

    print(
        f"TRAIN      | "
        f"R²={train_r2:.5f} | "
        f"RMSE={train_rmse:.5f} | "
        f"MAE={train_mae:.5f} | "
        f"Bias={train_bias:.5f}"
    )

    print(
        f"VALIDATION | "
        f"R²={val_r2:.5f} | "
        f"RMSE={val_rmse:.5f} | "
        f"MAE={val_mae:.5f} | "
        f"Bias={val_bias:.5f}"
    )

    results.append([
        fold,
        train_r2,
        val_r2,
        train_rmse,
        val_rmse,
        train_mae,
        val_mae,
        train_bias,
        val_bias
    ])

cv_metrics_df = pd.DataFrame(
    results,
    columns=[
        "Fold",
        "Train_R2",
        "Val_R2",
        "Train_RMSE",
        "Val_RMSE",
        "Train_MAE",
        "Val_MAE",
        "Train_Bias",
        "Val_Bias"
    ]
)

print("\n")
print(cv_metrics_df)


summary_df = pd.DataFrame({

    "Metric": [
        "R2",
        "RMSE",
        "MAE",
        "Bias"
    ],

    "Train_Mean": [

        cv_metrics_df["Train_R2"].mean(),
        cv_metrics_df["Train_RMSE"].mean(),
        cv_metrics_df["Train_MAE"].mean(),
        cv_metrics_df["Train_Bias"].mean()
    ],

    "Validation_Mean": [

        cv_metrics_df["Val_R2"].mean(),
        cv_metrics_df["Val_RMSE"].mean(),
        cv_metrics_df["Val_MAE"].mean(),
        cv_metrics_df["Val_Bias"].mean()
    ]
})

print(summary_df)


# ============================================================
# CV Mean ± Std Summary
# ============================================================

cv_stats_df = pd.DataFrame({

    "Metric": [
        "R2",
        "RMSE",
        "MAE",
        "Bias"
    ],

    "Train_Mean": [

        cv_metrics_df["Train_R2"].mean(),
        cv_metrics_df["Train_RMSE"].mean(),
        cv_metrics_df["Train_MAE"].mean(),
        cv_metrics_df["Train_Bias"].mean()
    ],

    "Train_SD": [

        cv_metrics_df["Train_R2"].std(),
        cv_metrics_df["Train_RMSE"].std(),
        cv_metrics_df["Train_MAE"].std(),
        cv_metrics_df["Train_Bias"].std()
    ],

    "Validation_Mean": [

        cv_metrics_df["Val_R2"].mean(),
        cv_metrics_df["Val_RMSE"].mean(),
        cv_metrics_df["Val_MAE"].mean(),
        cv_metrics_df["Val_Bias"].mean()
    ],

    "Validation_SD": [

        cv_metrics_df["Val_R2"].std(),
        cv_metrics_df["Val_RMSE"].std(),
        cv_metrics_df["Val_MAE"].std(),
        cv_metrics_df["Val_Bias"].std()
    ]
})

print("\nCross-Validation Mean ± SD")
print(cv_stats_df)


Fold 1
TRAIN      | R²=0.75569 | RMSE=0.16403 | MAE=0.11568 | Bias=0.00042
VALIDATION | R²=0.74266 | RMSE=0.16837 | MAE=0.11779 | Bias=0.00018

Fold 2
TRAIN      | R²=0.75580 | RMSE=0.16401 | MAE=0.11568 | Bias=0.00042
VALIDATION | R²=0.74273 | RMSE=0.16833 | MAE=0.11774 | Bias=0.00049


   Fold  Train_R2    Val_R2  Train_RMSE  Val_RMSE  Train_MAE   Val_MAE  \
0     1  0.755693  0.742660    0.164034  0.168365   0.115681  0.117792   
1     2  0.755800  0.742733    0.164010  0.168329   0.115677  0.117741   

   Train_Bias  Val_Bias  
0    0.000417  0.000184  
1    0.000416  0.000488  
  Metric  Train_Mean  Validation_Mean
0     R2    0.755747         0.742696
1   RMSE    0.164022         0.168347
2    MAE    0.115679         0.117766
3   Bias    0.000416         0.000336

Cross-Validation Mean ± SD
  Metric  Train_Mean      Train_SD  Validation_Mean  Validation_SD
0     R2    0.755747  7.535857e-05         0.742696       0.000052
1   RMSE    0.164022  1.711580e-05         0.168347      

In [14]:
# --------------- 8 ------------------



final_model = XGBRegressor(
    objective=underestimation_loss,       
    tree_method="gpu_hist",
    predictor="gpu_predictor",
    random_state=42,
    n_jobs=-1,
    **best_model_params
)

final_model.fit(
    X_train,
    y_train,
    verbose=False
)

,objective,<function und...x7f4d96275cf0>
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.963930837858085
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None
,feature_types,None


In [15]:

# -----------------------------
# Train Predictions
# -----------------------------
y_train_pred = final_model.predict(X_train)

# -----------------------------
# Train Metrics
# -----------------------------
train_r2   = r2_score(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae  = mean_absolute_error(y_train, y_train_pred)
train_bias = np.mean(y_train_pred - y_train)

# -----------------------------
# Print Results
# -----------------------------
print("\n==============================")
print("Training Performance")
print("==============================")
print(f"Train R²   : {train_r2:.4f}")
print(f"Train RMSE : {train_rmse:.4f}")
print(f"Train MAE  : {train_mae:.4f}")
print(f"Train Bias : {train_bias:.4f}")


Training Performance
Train R²   : 0.7791
Train RMSE : 0.1560
Train MAE  : 0.1096
Train Bias : 0.0004


In [16]:
# ------------------ 9 ----------------------

import joblib

# ============================================================
# Save Final Model
# ============================================================

MODEL_PATH = "xgboost_BoB_NO_SSH_model.pkl"

joblib.dump(
    final_model,
    MODEL_PATH
)

print("\n" + "="*60)
print("MODEL SAVED SUCCESSFULLY")
print("="*60)
print(f"Saved as: {MODEL_PATH}")


MODEL SAVED SUCCESSFULLY
Saved as: xgboost_BoB_NO_SSH_model.pkl
